# Entrenamiento y Evaluación de Modelos - Proyecto FraudIQ

## Introducción

Este notebook representa la **segunda fase** del proyecto final de Aprendizaje Automático, **FraudIQ**. Partiendo del dataset limpio, preprocesado y balanceado obtenido en la etapa anterior, nuestro objetivo aquí es entrenar, comparar y evaluar el rendimiento de múltiples modelos de clasificación supervisada.

El proceso incluirá los siguientes pasos clave:
* La carga del dataset limpio.
* La separación de datos en conjuntos de entrenamiento y prueba.
* El escalado de características numéricas.
* El entrenamiento de los siguientes algoritmos de clasificación:
    * **Nearest Neighbors Classifier**
    * **Random Forest Classifier**
    * **Naive Bayes Classifier**
    * **SVM (Support Vector Machine) Classifier**
    * **Neural Network (MLP) Classifier**
    * **Gaussian Process Classifier (GPC)**
* La evaluación de cada modelo utilizando métricas relevantes para problemas de fraude (Precisión, Recall, F1-Score, y la curva ROC-AUC).

El propósito final es seleccionar el modelo con el mejor rendimiento para la detección de transacciones fraudulentas.

## Fuente del Dataset

El conjunto de datos que se utilizará en este notebook es el resultado del proceso de limpieza y preprocesamiento realizado en la primera fase del proyecto. El archivo de entrada es `fraudiq_dataset_limpio.csv`.

**Notebook de Limpieza y Preparación de Datos:** [Fase 1 - Data Cleaning](https://colab.research.google.com/drive/1V0C3TEQeikEfE_CIplfgAV-JWU9s6yiF?usp=sharing)

## Autores del proyecto

> REALIZADO POR DEYBBY ROSARIO 2024-0504 Y SARAH PEÑA 2024-0506

## Paso 1: Importación de Librerías

Como paso inicial, importamos todas las librerías y módulos necesarios para el desarrollo de este notebook. Estas herramientas nos permitirán manipular los datos, preprocesarlos, entrenar los modelos de clasificación y, finalmente, evaluar su rendimiento de manera rigurosa.

In [1]:
# --- Manipulación y Visualización de Datos ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Herramientas de Preprocesamiento y División de Datos ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- Modelos de Clasificación de scikit-learn ---
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from lightgbm import LGBMClassifier

# --- Métricas de Evaluación de Modelos ---
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

## Paso 2: Carga del Dataset Limpio

El primer paso en nuestro notebook de modelado es cargar el conjunto de datos `fraudiq_dataset_limpio.csv`, que fue el resultado final de toda la fase de limpieza y preparación.

In [2]:
df = pd.read_csv('fraudiq_dataset_limpio.csv')
print("Dataset cargado correctamente desde un archivo local.")

Dataset cargado correctamente desde un archivo local.


In [3]:
print(f"Dimensiones del dataset: {df.shape}")
display(df.head())

Dimensiones del dataset: (87285, 13)


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_day,day_of_week
0,207636.50,0.00,0.00,1092121.32,1299757.81,0,0,1,0,0,0,19,0
1,19117.48,0.00,0.00,609638.61,628756.09,0,0,1,0,0,0,10,0
2,16766.96,919567.72,936334.68,126753.55,109986.59,0,1,0,0,0,0,0,6
3,25288.82,45543.00,20254.18,0.00,0.00,0,0,0,0,1,0,11,2
4,94345.08,0.00,0.00,1244185.60,1338530.68,0,0,1,0,0,0,20,1


Paso 2: Separación de Características (X) y Variable Objetivo (y)

Antes de entrenar cualquier modelo, debemos dividir nuestro dataset en dos componentes esenciales:

1. X (Matriz de Características): Contiene todas las columnas que el modelo utilizará como entrada para aprender los patrones. En nuestro caso, son todas las columnas excepto isFraud.

2. y (Vector Objetivo): Contiene únicamente la columna que queremos predecir, es decir, isFraud.

In [4]:
# Verificar que el DataFrame 'df' existe antes de proceder

# 'y' es la columna que queremos predecir (nuestro objetivo)
y = df['isFraud']

# 'X' son todas las demás columnas que usaremos como predictoras
X = df.drop('isFraud', axis=1)

print("Separación completada exitosamente.")
print(f"Dimensiones de X (características): {X.shape}")
print(f"Dimensiones de y (objetivo): {y.shape}")

Separación completada exitosamente.
Dimensiones de X (características): (87285, 12)
Dimensiones de y (objetivo): (87285,)


In [5]:
print("\nPrimeras 5 filas de X (características):")
display(X.head())


Primeras 5 filas de X (características):


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_day,day_of_week
0,207636.50,0.00,0.00,1092121.32,1299757.81,0,1,0,0,0,19,0
1,19117.48,0.00,0.00,609638.61,628756.09,0,1,0,0,0,10,0
2,16766.96,919567.72,936334.68,126753.55,109986.59,1,0,0,0,0,0,6
3,25288.82,45543.00,20254.18,0.00,0.00,0,0,0,1,0,11,2
4,94345.08,0.00,0.00,1244185.60,1338530.68,0,1,0,0,0,20,1


In [6]:
print("\nPrimeras 20 filas de y (objetivo):")
display(y.head(20))


Primeras 20 filas de y (objetivo):


,isFraud
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,0
9,1


## Paso 3: División de Datos en Conjuntos de Entrenamiento y Prueba

Ahora dividiremos nuestros datos (`X` y `y`) en dos conjuntos separados:

* **Conjunto de Entrenamiento (Training Set):** Corresponde a la mayor parte de los datos (generalmente 70-80%). El modelo "aprenderá" los patrones de fraude utilizando exclusivamente este conjunto.
* **Conjunto de Prueba (Test Set):** Es una porción más pequeña (20-30%) que el modelo **nunca verá** durante el entrenamiento. Usaremos este conjunto al final para evaluar qué tan bien generaliza nuestro modelo a datos nuevos y desconocidos.

Es fundamental realizar esta división para obtener una medida honesta y no sesgada del rendimiento del modelo.

**Nota Importante:** Usaremos la **estratificación (`stratify=y`)** al dividir. Esto garantiza que la proporción de transacciones fraudulentas y no fraudulentas sea exactamente la misma tanto en el conjunto de entrenamiento como en el de prueba, lo cual es vital para una evaluación correcta.

In [7]:
# División de Datos en Entrenamiento y Prueba
'''
Usamos la función train_test_split de scikit-learn
test_size=0.3 significa que el 30% de los datos se usará para pruebas y el 70% para entrenamiento.
random_state=42 asegura que la división sea la misma cada vez que ejecutamos el código.
stratify=y asegura que la proporción de clases se mantenga en ambos conjuntos.
'''
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Verificación de la División
print("División de datos completada.")
print("\nDimensiones de los conjuntos de datos:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\nVerificación de la estratificación (proporción de fraude):")
print(f"Distribución en y_train:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\nDistribución en y_test:\n{y_test.value_counts(normalize=True).round(4)}")

División de datos completada.

Dimensiones de los conjuntos de datos:
X_train: (61099, 12)
X_test:  (26186, 12)
y_train: (61099,)
y_test:  (26186,)

Verificación de la estratificación (proporción de fraude):
Distribución en y_train:
isFraud
0    0.9091
1    0.0909
Name: proportion, dtype: float64

Distribución en y_test:
isFraud
0    0.9091
1    0.0909
Name: proportion, dtype: float64


## Paso 4: Escalado de Características (Feature Scaling)

Muchos algoritmos de machine learning, especialmente los que se basan en distancias como **SVM** y **Nearest Neighbors**, o los que usan descenso de gradiente como las **Redes Neuronales**, son sensibles a la escala de las características de entrada. Una variable con un rango muy grande (como `amount`, que puede llegar a millones) podría dominar el proceso de aprendizaje sobre una variable con un rango pequeño (como `hour_of_day`, de 0 a 23).

Para evitar esto y asegurar que todas las características contribuyan de manera equitativa, aplicaremos un **escalado de características**. Utilizaremos `StandardScaler`, que transforma los datos para que tengan una media de 0 y una desviación estándar de 1.

### Metodología Crítica (Para Evitar Fuga de Datos)

* Ajustamos (`fit`) el `StandardScaler` **únicamente con los datos de entrenamiento (`X_train`)**. Esto calcula la media y la desviación estándar necesarias para el escalado.
* Luego, usamos ese mismo scaler ya ajustado para **transformar** tanto el conjunto de entrenamiento (`X_train`) como el de prueba (`X_test`).

Este procedimiento es vital para prevenir la **fuga de datos (data leakage)**, garantizando que nuestro conjunto de prueba siga siendo una verdadera representación de datos no vistos.

In [8]:
# --- Escalado de Características Numéricas ---

# Identificar las columnas a escalar (todas excepto las que son binarias del one-hot encoding)
# Las columnas 'type_*' ya están en una escala de 0 a 1, por lo que no necesitan ser escaladas.
columns_to_scale = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'hour_of_day', 'day_of_week'
]

# Crear una instancia del StandardScaler
scaler = StandardScaler()

# Ajustar el scaler CON y transformar SOLO el conjunto de entrenamiento
X_train[columns_to_scale] = scaler.fit_transform(X_train[columns_to_scale])

# Usar el scaler YA AJUSTADO para transformar el conjunto de prueba
X_test[columns_to_scale] = scaler.transform(X_test[columns_to_scale])


# --- Verificación del Escalado ---
print("Escalado de características completado.")
print("\nVista previa de X_train después del escalado:")
display(X_train.head())

Escalado de características completado.

Vista previa de X_train después del escalado:


,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_IN,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,hour_of_day,day_of_week
68885,-0.320116,-0.332780,-0.307088,-0.347406,-0.370117,0,0,0,1,0,1.118291,1.146677
67883,-0.002620,-0.325568,-0.190922,0.595114,0.409521,1,0,0,0,0,-0.193439,-1.167364
4835,-0.314765,-0.322344,-0.301958,-0.347406,-0.370117,0,0,0,1,0,0.899669,1.609485
46762,-0.251216,0.149015,0.215409,-0.226498,-0.351675,1,0,0,0,0,-0.412060,-0.704556
70760,-0.296377,-0.332902,-0.307088,-0.347406,-0.370117,0,0,0,1,0,-0.193439,-1.167364


## Paso 5: Entrenamiento y Evaluación de Modelos

Con los datos ya preparados, procedemos a la fase de modelado. El proceso consistirá en:

1.  Definir una lista con los seis modelos de clasificación que vamos a probar.
2.  Iterar sobre cada modelo para entrenarlo con nuestro conjunto de datos de entrenamiento (`X_train`, `y_train`).
3.  Evaluar el rendimiento de cada modelo entrenado utilizando el conjunto de prueba (`X_test`, `y_test`) y un conjunto de métricas clave.
4.  Almacenar los resultados en una estructura de datos para su posterior comparación.

In [9]:
# Preparar un diccionario para almacenar todos los resultados
results = {}

# --- Lote 1: Nearest Neighbors, Random Forest y Logistic Regression ---
print("--- Iniciando Lote 1 de Entrenamiento ---")
models_batch_1 = {
    "Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000)
}

for model_name, model in models_batch_1.items():
    print(f"\nEntrenando: {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_pred_proba)
    }
    print(f"{model_name} evaluado exitosamente.")

--- Iniciando Lote 1 de Entrenamiento ---

Entrenando: Nearest Neighbors...
Nearest Neighbors evaluado exitosamente.

Entrenando: Random Forest...
Random Forest evaluado exitosamente.

Entrenando: Logistic Regression...
Logistic Regression evaluado exitosamente.


In [10]:
# --- Lote 2: Naive Bayes y SVM ---
print("\n--- Iniciando Lote 2 de Entrenamiento ---")
models_batch_2 = {
    "Gaussian Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42, probability=True)
}

for model_name, model in models_batch_2.items():
    print(f"\nEntrenando: {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_pred_proba)
    }
    print(f"{model_name} evaluado exitosamente.")


--- Iniciando Lote 2 de Entrenamiento ---

Entrenando: Gaussian Naive Bayes...
Gaussian Naive Bayes evaluado exitosamente.

Entrenando: SVM...
SVM evaluado exitosamente.


In [11]:
# --- Lote 3: Red Neuronal y LightGBM ---
print("\n--- Iniciando Lote 3 de Entrenamiento ---")
models_batch_3 = {
    "Neural Network (MLP)": MLPClassifier(random_state=42, max_iter=1000),
    "LightGBM": LGBMClassifier(random_state=42)
}

for model_name, model in models_batch_3.items():
    print(f"\nEntrenando: {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_pred_proba)
    }
    print(f"{model_name} evaluado exitosamente.")

print("\n--- Proceso de evaluación completado para todos los lotes. ---")


--- Iniciando Lote 3 de Entrenamiento ---

Entrenando: Neural Network (MLP)...
Neural Network (MLP) evaluado exitosamente.

Entrenando: LightGBM...
[LightGBM] [Info] Number of positive: 5554, number of negative: 55545
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001998 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1318
[LightGBM] [Info] Number of data points in the train set: 61099, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.090902 -> initscore=-2.302675
[LightGBM] [Info] Start training from score -2.302675
LightGBM evaluado exitosamente.

--- Proceso de evaluación completado para todos los lotes. ---


In [12]:
# --- Visualización de Resultados Consolidados ---
# Convertir el diccionario de resultados (que contiene los 6 modelos) a un DataFrame
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by="F1-Score", ascending=False)

print("\nTabla de Comparación de Rendimiento de los Modelos:")
display(results_df)


Tabla de Comparación de Rendimiento de los Modelos:


,Accuracy,Precision,Recall,F1-Score,ROC AUC
LightGBM,0.996869,0.976378,0.989500,0.982895,0.999167
Random Forest,0.993699,0.982158,0.947921,0.964736,0.998901
Neural Network (MLP),0.989498,0.960630,0.922302,0.941076,0.996200
SVM,0.976094,0.950693,0.777404,0.855360,0.978835
Nearest Neighbors,0.971053,0.896434,0.770685,0.828817,0.945829
Logistic Regression,0.969602,0.957828,0.696346,0.806420,0.982576
Gaussian Naive Bayes,0.518025,0.158393,0.997060,0.273361,0.916863


## Análisis y Elección del Modelo Ganador

Aquí tienes un desglose detallado de los resultados y por qué cada modelo se desempeñó como lo hizo.

### El Ganador Indiscutible: LightGBM
Este modelo es la elección clara y superior por varias razones clave, especialmente en el contexto de la detección de fraude:

* **F1-Score (0.9829):** El más alto de todos, indicando el mejor balance posible entre Precisión y Recall. Esta suele ser la métrica más importante para problemas desbalanceados.
* **Recall (0.9895):** ¡Excepcional! Esto significa que el modelo fue capaz de identificar correctamente el **98.9%** de todas las transacciones fraudulentas reales. En un escenario real, esto se traduce en minimizar las pérdidas económicas al no dejar pasar los fraudes.
* **Precision (0.9763):** También de primer nivel. Significa que cuando el modelo dijo que una transacción era fraudulenta, acertó el **97.6%** de las veces. Esto es crucial para la eficiencia operativa, ya que minimiza el tiempo que los analistas pierden investigando falsas alarmas.
* **ROC AUC (0.9991):** Un valor casi perfecto, lo que demuestra su excelente capacidad para distinguir entre clases en todos los umbrales de decisión.

### Menciones Honoríficas

**Random Forest:** Un segundo lugar muy sólido. Su **Precisión** es ligeramente superior a la de LightGBM, pero su **Recall** es un poco más bajo (94.7%). Sigue siendo un modelo excelente, pero LightGBM lo supera en la métrica más crítica para este problema: la capacidad de capturar la mayor cantidad de fraudes.

**Neural Network (MLP):** Un buen tercer lugar. Es un modelo robusto y de buen rendimiento, pero en este caso, fue superado por los modelos de ensamble (Random Forest y LightGBM), que a menudo destacan en datos tabulares como el nuestro.


### Análisis de los modelos con menos rendimiento

**SVM y Nearest Neighbors:** Su principal debilidad fue un **Recall** significativamente más bajo (~77%). Esto significa que, aunque eran razonablemente precisos cuando detectaban un fraude, dejaron pasar casi una de cada cuatro transacciones fraudulentas, lo cual es inaceptable para un sistema de detección de fraude.

**Logistic Regression:** Aunque es muy rápido, su bajo **Recall** (69.6%) lo descarta como una opción final. Demuestra que el problema tiene relaciones no lineales que este modelo no puede capturar eficazmente.

**Gaussian Naive Bayes:** Este modelo ilustra un caso clásico de "falso rendimiento". Su **Recall** es casi perfecto (99.7%), pero su **Precisión** es abismal (15.8%). En la práctica, esto significaría que para atrapar a casi todos los estafadores, el sistema inundaría a los analistas con una cantidad masiva de falsas alarmas (más de 4 de cada 5 alertas serían incorrectas), haciendo que el sistema sea inútil por el "ruido" que genera.

## Paso 6: Guardado de los Modelos Entrenados

Una vez entrenados y evaluados los modelos, el siguiente paso es guardarlos en un archivo. Este proceso, conocido como serialización o persistencia del modelo, nos permite cargar el modelo entrenado en el futuro para hacer predicciones sin necesidad de repetir todo el proceso de entrenamiento.

Nota: El siguiente código volverá a entrenar cada modelo brevemente antes de guardarlo para asegurar que tenemos la versión final lista para ser persistida.

In [13]:
# Importar la librería para guardar los modelos
import pickle

models = {
    "Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42, probability=True),
    "Neural Network (MLP)": MLPClassifier(random_state=42, max_iter=1000),
    "LightGBM": LGBMClassifier(random_state=42)
}

print("Iniciando el proceso de guardado de modelos con prefijo 'FraudIQ_'...")

# Iteramos sobre el diccionario para entrenar y guardar cada modelo
for model_name, model in models.items():
    print(f"\n--- Entrenando y guardando: {model_name} ---")

    # Entrenar el modelo
    model.fit(X_train, y_train)

    # Definir el nombre del archivo
    filename = f'FraudIQ_{model_name.replace(" ", "_").replace("(", "").replace(")", "")}.pkl'

    # Guardar el modelo en un archivo .pkl usando pickle
    with open(filename, 'wb') as file:
        pickle.dump(model, file)

    print(f"Modelo '{model_name}' guardado exitosamente como '{filename}'")

print("\n--- Proceso de guardado completado para todos los modelos. ---")

Iniciando el proceso de guardado de modelos con prefijo 'FraudIQ_'...

--- Entrenando y guardando: Nearest Neighbors ---
Modelo 'Nearest Neighbors' guardado exitosamente como 'FraudIQ_Nearest_Neighbors.pkl'

--- Entrenando y guardando: Random Forest ---
Modelo 'Random Forest' guardado exitosamente como 'FraudIQ_Random_Forest.pkl'

--- Entrenando y guardando: Gaussian Naive Bayes ---
Modelo 'Gaussian Naive Bayes' guardado exitosamente como 'FraudIQ_Gaussian_Naive_Bayes.pkl'

--- Entrenando y guardando: SVM ---
Modelo 'SVM' guardado exitosamente como 'FraudIQ_SVM.pkl'

--- Entrenando y guardando: Neural Network (MLP) ---
Modelo 'Neural Network (MLP)' guardado exitosamente como 'FraudIQ_Neural_Network_MLP.pkl'

--- Entrenando y guardando: LightGBM ---
[LightGBM] [Info] Number of positive: 5554, number of negative: 55545
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005537 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM

In [14]:
# --- Verificación: Cargar el mejor modelo y probarlo con 10 muestras aleatorias ---

filename_to_load = 'FraudIQ_LightGBM.pkl'
print(f"Verificando el guardado al cargar el mejor modelo ({filename_to_load})...")

try:
    # Cargar el modelo desde el archivo
    with open(filename_to_load, 'rb') as file:
        loaded_lightgbm_model = pickle.load(file)
    print("Modelo cargado exitosamente.")

    # Tomar 10 muestras aleatorias del conjunto de prueba
    sample_X = X_test.sample(n=20, random_state=42) # random_state para que la muestra sea siempre la misma

    # Obtener las etiquetas verdaderas correspondientes a esas muestras
    sample_y_true = y_test.loc[sample_X.index]

    # Realizar predicciones en las muestras
    predictions = loaded_lightgbm_model.predict(sample_X)

    # Crear un DataFrame para comparar los resultados
    verification_df = pd.DataFrame({
        'Valor Real': sample_y_true,
        'Predicción del Modelo': predictions
    })

    # Mapear los valores 0 y 1 a etiquetas más descriptivas
    verification_df['Valor Real'] = verification_df['Valor Real'].map({0: 'No Fraude', 1: 'Fraude'})
    verification_df['Predicción del Modelo'] = verification_df['Predicción del Modelo'].map({0: 'No Fraude', 1: 'Fraude'})


    print("\nResultados de la predicción en 10 filas aleatorias:")
    display(verification_df)

except FileNotFoundError:
    print(f"Error al verificar: No se encontró el archivo '{filename_to_load}'.")

Verificando el guardado al cargar el mejor modelo (FraudIQ_LightGBM.pkl)...
Modelo cargado exitosamente.

Resultados de la predicción en 10 filas aleatorias:


,Valor Real,Predicción del Modelo
63589,No Fraude,No Fraude
29292,No Fraude,No Fraude
29034,No Fraude,No Fraude
76243,No Fraude,No Fraude
47613,No Fraude,No Fraude
7751,No Fraude,No Fraude
35499,Fraude,Fraude
976,No Fraude,No Fraude
30535,No Fraude,No Fraude
17716,No Fraude,No Fraude


## Paso 7: Consolidación Final - Entrenamiento, Evaluación y Guardado de Pipelines

Llegamos al paso final que unifica todo nuestro trabajo. En esta sección, implementaremos un flujo de trabajo profesional utilizando Pipelines de scikit-learn. Este enfoque nos permite encapsular el preprocesamiento y el modelado en un solo objeto, garantizando consistencia, previniendo la fuga de datos y preparando nuestros modelos para un despliegue real.

### 7.1. Definición del Preprocesador Común
Para asegurar que todos los modelos sean evaluados de manera justa y que el preprocesamiento sea idéntico en cada caso, primero definimos un ColumnTransformer. Este objeto se encargará de aplicar los pasos de transformación necesarios (en este caso, la imputación de valores faltantes y el StandardScaler) únicamente a las columnas numéricas, dejando el resto sin cambios.

In [15]:
# Importaciones clave para este paso
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import joblib

# --- 1. Definir el Preprocesador Común ---

numeric_features = [
    'amount', 'oldbalanceOrg', 'newbalanceOrig',
    'oldbalanceDest', 'newbalanceDest', 'hour_of_day', 'day_of_week'
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[('num', numeric_transformer, numeric_features)],
    remainder='passthrough'
)

### 7.2. Configuración de Modelos y Almacenamiento

A continuación, creamos un diccionario que contiene las instancias de los seis modelos base que deseamos comparar. Adicionalmente, preparamos dos diccionarios vacíos: uno para almacenar las métricas de rendimiento de cada pipeline y otro para guardar el objeto del pipeline ya entrenado.

In [16]:
# --- 2. Definir los Modelos y Preparar Almacenamiento ---
# Diccionario con las instancias de los modelos base
base_models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Nearest Neighbors": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gaussian Naive Bayes": GaussianNB(),
    "SVM": SVC(random_state=42, probability=True),
    "LightGBM": LGBMClassifier(random_state=42)
}

# Diccionarios para guardar los resultados y los pipelines entrenados
pipeline_results = {}
trained_pipelines = {}

### 7.3. Bucle de Entrenamiento y Evaluación de Pipelines

Este es el núcleo del proceso. Iteramos a través de cada modelo base y, en cada ciclo, creamos un Pipeline completo que une el preprocesador común con el modelo actual. Este pipeline se entrena y se evalúa con una sola llamada, asegurando un flujo de trabajo limpio. Al finalizar el bucle, los resultados de rendimiento se presentan en una tabla comparativa para identificar el modelo ganador.

In [17]:
# --- 3. Bucle de Entrenamiento, Evaluación y Almacenamiento de Pipelines ---
print("Iniciando el ciclo final de entrenamiento y evaluación de pipelines...")

for model_name, model in base_models.items():
    print(f"--- Procesando pipeline para: {model_name} ---")

    # Crear el pipeline final uniendo el preprocesador con el modelo actual
    final_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])

    # Entrenar el pipeline
    final_pipeline.fit(X_train, y_train)

    # Evaluar el pipeline
    y_pred = final_pipeline.predict(X_test)
    y_pred_proba = final_pipeline.predict_proba(X_test)[:, 1]

    # Guardar los resultados de la evaluación
    pipeline_results[model_name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_pred_proba)
    }

    # Guardar el pipeline entrenado en nuestro diccionario
    trained_pipelines[model_name] = final_pipeline
    print(f"Pipeline para '{model_name}' entrenado y evaluado.")

print("\n--- Ciclo de evaluación completado. ---")

Iniciando el ciclo final de entrenamiento y evaluación de pipelines...
--- Procesando pipeline para: Logistic Regression ---
Pipeline para 'Logistic Regression' entrenado y evaluado.
--- Procesando pipeline para: Nearest Neighbors ---
Pipeline para 'Nearest Neighbors' entrenado y evaluado.
--- Procesando pipeline para: Random Forest ---
Pipeline para 'Random Forest' entrenado y evaluado.
--- Procesando pipeline para: Gaussian Naive Bayes ---
Pipeline para 'Gaussian Naive Bayes' entrenado y evaluado.
--- Procesando pipeline para: SVM ---
Pipeline para 'SVM' entrenado y evaluado.
--- Procesando pipeline para: LightGBM ---
[LightGBM] [Info] Number of positive: 5554, number of negative: 55545
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001816 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1318
[LightGBM] [Info] Number of data points in

/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Pipeline para 'LightGBM' entrenado y evaluado.

--- Ciclo de evaluación completado. ---


### 7.4. Guardado de los Pipelines Entrenados para Despliegue

Como paso final, recorremos el diccionario que contiene los pipelines ya entrenados y guardamos cada uno de estos objetos en un archivo .pkl individual. Estos archivos son los artefactos finales y más valiosos de nuestro proyecto, ya que encapsulan toda la lógica de preprocesamiento y el modelo entrenado, listos para ser cargados en una aplicación para hacer predicciones sobre datos nuevos.

In [18]:
# --- 4. Visualización de Resultados de los Pipelines ---
results_df_pipelines = pd.DataFrame(pipeline_results).T
results_df_pipelines = results_df_pipelines.sort_values(by="F1-Score", ascending=False)

print("\nTabla de Comparación de Rendimiento de los Pipelines:")
display(results_df_pipelines)


Tabla de Comparación de Rendimiento de los Pipelines:


,Accuracy,Precision,Recall,F1-Score,ROC AUC
LightGBM,0.996869,0.976378,0.989500,0.982895,0.999167
Random Forest,0.993470,0.979185,0.948341,0.963516,0.998729
SVM,0.976094,0.950693,0.777404,0.855360,0.978835
Nearest Neighbors,0.971053,0.896434,0.770685,0.828817,0.945829
Logistic Regression,0.969602,0.957828,0.696346,0.806420,0.982576
Gaussian Naive Bayes,0.518025,0.158393,0.997060,0.273361,0.916863


In [19]:
# --- 5. Guardado de los Pipelines Entrenados en Archivos ---
print("\n--- Guardando los pipelines entrenados en archivos .pkl ---")

for model_name, pipeline_object in trained_pipelines.items():
    filename = f'FraudIQ_Pipeline_{model_name.replace(" ", "_").replace("(", "").replace(")", "")}.pkl'
    joblib.dump(pipeline_object, filename)
    print(f"Pipeline '{model_name}' guardado exitosamente como '{filename}'")

print("\n--- Proceso finalizado. Todos los pipelines están guardados y listos para el despliegue. ---")


--- Guardando los pipelines entrenados en archivos .pkl ---
Pipeline 'Logistic Regression' guardado exitosamente como 'FraudIQ_Pipeline_Logistic_Regression.pkl'
Pipeline 'Nearest Neighbors' guardado exitosamente como 'FraudIQ_Pipeline_Nearest_Neighbors.pkl'
Pipeline 'Random Forest' guardado exitosamente como 'FraudIQ_Pipeline_Random_Forest.pkl'
Pipeline 'Gaussian Naive Bayes' guardado exitosamente como 'FraudIQ_Pipeline_Gaussian_Naive_Bayes.pkl'
Pipeline 'SVM' guardado exitosamente como 'FraudIQ_Pipeline_SVM.pkl'
Pipeline 'LightGBM' guardado exitosamente como 'FraudIQ_Pipeline_LightGBM.pkl'

--- Proceso finalizado. Todos los pipelines están guardados y listos para el despliegue. ---


## Conclusión General

En este proyecto, se ha completado un ciclo de vida de machine learning de extremo a extremo, transformando un conjunto de datos crudo y masivo en un modelo de clasificación de alto rendimiento, listo para su aplicación.

Mediante un riguroso proceso de limpieza, ingeniería de características y un submuestreo estratificado estratégico, se construyó un dataset balanceado y representativo. Sobre este, se entrenaron y evaluaron sistemáticamente seis algoritmos de clasificación, midiendo su eficacia con métricas clave para la detección de fraude.

El análisis comparativo demostró la clara superioridad del modelo LightGBM, el cual alcanzó un excepcional equilibrio entre un alto Recall (98.9%) y una alta Precisión (97.6%). Este resultado lo posiciona como la solución óptima para identificar transacciones fraudulentas minimizando tanto las pérdidas económicas como las falsas alarmas.

Finalmente, para asegurar la máxima robustez y facilitar la producción, todo el flujo de trabajo (preprocesamiento y modelo) fue encapsulado en un Pipeline de scikit-learn. Este objeto final fue guardado exitosamente, culminando el proyecto con un artefacto tangible y profesional (FraudIQ_Pipeline_LGBM.pkl), preparado para su despliegue en la aplicación web de nuestro producto FraudIQ.

## FIN DEL DOCUMENTO

> REALIZADO POR DEYBBY ROSARIO 2024-0504 Y SARAH PEÑA 2024-0506